# Entities

Explore and manage knowledge base entities.

## Setup

Install the package and connect to the database.

Add your `DATABASE_URL` to Colab Secrets (the key icon in the left sidebar) before running.

In [ ]:
!pip install -q "git+https://github.com/ContextNews/context-db.git" pandas

In [ ]:
import os
from google.colab import userdata

os.environ['DATABASE_URL'] = userdata.get('DATABASE_URL')

import pandas as pd
from sqlalchemy import text
from context_db.connection import get_session

print("Connected.")

---
## 1. Unresolved entities

Entities extracted from articles that have not been matched to a KB entry, ordered by how often they appear.

In [ ]:
NER_TYPE = "ALL"  # ALL | PERSON | ORG | GPE | LOC | FAC | NORP | ...

with get_session() as session:
    ner_filter = "AND aem.ner_type = :ner_type" if NER_TYPE != "ALL" else ""
    params = {"ner_type": NER_TYPE} if NER_TYPE != "ALL" else {}

    rows = session.execute(text(f"""
        SELECT
            aem.mention_text,
            aem.ner_type,
            SUM(aem.mention_count) AS total_count
        FROM article_entity_mentions aem
        LEFT JOIN kb_entity_aliases kea
            ON UPPER(aem.mention_text) = UPPER(kea.alias)
        WHERE kea.alias IS NULL
        {ner_filter}
        GROUP BY aem.mention_text, aem.ner_type
        ORDER BY total_count DESC
    """), params).all()

pd.DataFrame(rows, columns=["mention_text", "ner_type", "total_count"])

---
## 2. Look up entity by alias

Exact match first, then falls back to partial (ILIKE) if not found.

In [ ]:
ALIAS = "Joe Biden"

with get_session() as session:
    row = session.execute(text("""
        SELECT ke.qid, ke.entity_type, ke.name, ke.description
        FROM kb_entity_aliases kea
        JOIN kb_entities ke ON ke.qid = kea.qid
        WHERE UPPER(kea.alias) = UPPER(:name)
        LIMIT 1
    """), {"name": ALIAS}).one_or_none()

    if not row:
        matches = session.execute(text("""
            SELECT DISTINCT ke.qid, ke.entity_type, ke.name, ke.description
            FROM kb_entity_aliases kea
            JOIN kb_entities ke ON ke.qid = kea.qid
            WHERE kea.alias ILIKE :pattern
            ORDER BY ke.name
            LIMIT 20
        """), {"pattern": f"%{ALIAS}%"}).all()

        if not matches:
            print(f"No entity found for: {ALIAS!r}")
            row = None
        elif len(matches) == 1:
            row = matches[0]
        else:
            print(f"No exact match. Partial matches for {ALIAS!r}:")
            display(pd.DataFrame(matches, columns=["qid", "entity_type", "name", "description"]))
            row = None

    if row:
        aliases = session.execute(text("""
            SELECT alias FROM kb_entity_aliases
            WHERE qid = :qid
            ORDER BY alias
        """), {"qid": row.qid}).scalars().all()

if row:
    print(f"QID:         {row.qid}")
    print(f"Type:        {row.entity_type}")
    print(f"Name:        {row.name}")
    if row.description:
        print(f"Description: {row.description}")
    print(f"\nAliases ({len(aliases)}):")
    for alias in aliases:
        print(f"  {alias}")

---
## 3. Add a location entity

In [ ]:
QID           = "Q84"
NAME          = "London"
DESCRIPTION   = "Capital city of England and the United Kingdom"
LOCATION_TYPE = "city"    # city | country | region | etc.
COUNTRY_CODE  = "GB"      # ISO 3166-1 alpha-2, or None
LAT           = 51.5074   # or None
LON           = -0.1278   # or None
ALIASES       = ["London, UK", "Greater London"]  # extra aliases beyond NAME

with get_session() as session:
    session.execute(text("""
        INSERT INTO kb_entities (qid, entity_type, name, description)
        VALUES (:qid, 'location', :name, :description)
    """), {"qid": QID, "name": NAME, "description": DESCRIPTION})

    if LAT is not None and LON is not None:
        session.execute(text("""
            INSERT INTO kb_locations (qid, location_type, country_code, coordinates)
            VALUES (:qid, :location_type, :country_code,
                    ST_SetSRID(ST_MakePoint(:lon, :lat), 4326)::geography)
        """), {"qid": QID, "location_type": LOCATION_TYPE, "country_code": COUNTRY_CODE,
               "lat": LAT, "lon": LON})
    else:
        session.execute(text("""
            INSERT INTO kb_locations (qid, location_type, country_code, coordinates)
            VALUES (:qid, :location_type, :country_code, NULL)
        """), {"qid": QID, "location_type": LOCATION_TYPE, "country_code": COUNTRY_CODE})

    all_aliases = [NAME] + [a for a in ALIASES if a != NAME]
    for alias in all_aliases:
        session.execute(text("""
            INSERT INTO kb_entity_aliases (alias, qid) VALUES (:alias, :qid)
            ON CONFLICT DO NOTHING
        """), {"alias": alias, "qid": QID})

    session.commit()
    print(f"Added location {NAME!r} ({QID}) with {len(all_aliases)} aliases.")

---
## 4. Add a person entity

In [ ]:
QID           = "Q76"
NAME          = "Barack Obama"
DESCRIPTION   = "44th president of the United States"
NATIONALITIES = ["US"]   # ISO codes, or []
ALIASES       = ["Obama", "Barack H. Obama"]  # extra aliases beyond NAME

with get_session() as session:
    session.execute(text("""
        INSERT INTO kb_entities (qid, entity_type, name, description)
        VALUES (:qid, 'person', :name, :description)
    """), {"qid": QID, "name": NAME, "description": DESCRIPTION})

    session.execute(text("""
        INSERT INTO kb_persons (qid, nationalities)
        VALUES (:qid, :nationalities)
    """), {"qid": QID, "nationalities": NATIONALITIES})

    all_aliases = [NAME] + [a for a in ALIASES if a != NAME]
    for alias in all_aliases:
        session.execute(text("""
            INSERT INTO kb_entity_aliases (alias, qid) VALUES (:alias, :qid)
            ON CONFLICT DO NOTHING
        """), {"alias": alias, "qid": QID})

    session.commit()
    print(f"Added person {NAME!r} ({QID}) with {len(all_aliases)} aliases.")

---
## 5. Add alias to existing entity

In [ ]:
QID   = "Q76"
ALIAS = "Barry Obama"

with get_session() as session:
    entity = session.execute(text("""
        SELECT name FROM kb_entities WHERE qid = :qid
    """), {"qid": QID}).one_or_none()

    if not entity:
        print(f"No entity found with QID {QID!r}")
    else:
        existing = session.execute(text("""
            SELECT 1 FROM kb_entity_aliases WHERE alias = :alias AND qid = :qid
        """), {"alias": ALIAS, "qid": QID}).one_or_none()

        if existing:
            print(f"Alias {ALIAS!r} already exists for {entity.name} ({QID})")
        else:
            session.execute(text("""
                INSERT INTO kb_entity_aliases (alias, qid) VALUES (:alias, :qid)
            """), {"alias": ALIAS, "qid": QID})
            session.commit()
            print(f"Added alias {ALIAS!r} to {entity.name} ({QID})")